In [ ]:
import pandas as pd
import os
os.chdir('file_directory')

In [ ]:
os.getcwd()

In [ ]:
# Reading in the file
list = pd.read_csv("jcb_202104100_tables2.csv", header=1)

# Accessing the necessary columns
data_1 = list.iloc[:,:3]

# You can access biomaRt using the GUI @ https://useast.ensembl.org/biomart/martview/c65bbca2812d78565e1369ddff0e327f
# It offers a lot of functions. (Here, I tried viewing the human protein and gene homologs to that of yeast)

# Load the data you downloaded from biomaRt
master_list = pd.read_table("mart_export (1).txt")

# Merge DataFrames on the ORF columns
merged_df = pd.merge(data_1, master_list, left_on='ORF', right_on='Gene stable ID', how='inner')
# Here I tried to merge the dataframes based on the commonalities from data_1. 
# If you want to preserve data in data_1 which don't have a match, you can use `how='left'` 

# Display the merged DataFrame
merged_df

In [ ]:
# If you want to pull out specific columns
merge = merged_df[["Gene/ORF", "ORF", "Description", "Human gene name" , "Human protein or transcript stable ID", "Human gene stable ID"]]

# Save it to a csv based on the need
merge.to_csv('Human_Homologs_table2.csv', index=False)

In order to do a deep dive into the identified human gene/protein homologs, I used the Gene Cards webiste @ https://genealacart.genecards.org/Query. Here you can upload 10 Genes at a time and see all the associated attributes of it (Localizations, Phenotypes, Pathways, Gene Summaries, UNIPROT Summaries, etc.,)

In [ ]:
# If you have a bunch of files, that you want to bring in together, you can use this
### Make sure that all these files have the same tab names and the same columns ###

# List of Excel file paths
excel_files = [
    'GeneALaCart-5201230-240605-201600.xlsx',
    'GeneALaCart-5201230-240605-201631.xlsx',
    'GeneALaCart-5201230-240605-201738.xlsx',
    'GeneALaCart-5201230-240605-201800.xlsx',
    'GeneALaCart-5201230-240605-201830.xlsx',
    'GeneALaCart-5201230-240605-203138.xlsx',
    'GeneALaCart-5201230-240605-203207.xlsx',
    'GeneALaCart-5201230-240605-203231.xlsx',
    'GeneALaCart-5201230-240605-204458.xlsx',
    'GeneALaCart-5201230-240605-220900.xlsx'
    # Add paths to all 8 files
]

# Output file path
output_file = 'Gene_Cards_Human_Homologs.xlsx'

# Initialize a dictionary to hold dataframes for each sheet
all_sheets = {}

# Iterate over each file
for file in excel_files:
    # Load the Excel file
    xls = pd.ExcelFile(file)
    
    # Iterate over each sheet in the file
    for sheet_name in xls.sheet_names:
        # Read the sheet into a dataframe
        df = xls.parse(sheet_name)
        
        # Append the dataframe to the list of dataframes for this sheet
        if sheet_name not in all_sheets:
            all_sheets[sheet_name] = []
        all_sheets[sheet_name].append(df)

# Create a Pandas Excel writer using openpyxl as the engine
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Iterate over each sheet
    for sheet_name, dfs in all_sheets.items():
        # Concatenate all dataframes for this sheet
        merged_df = pd.concat(dfs, ignore_index=True)
        
        # Write the merged dataframe to the output file
        merged_df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f'Merged Excel file saved as {output_file}')